In [18]:
from typing import TypedDict, Literal, Any
import csv
import json
import os
import re
from collections import Counter
from datetime import datetime
from statistics import mean
import pandas as pd                                                                                                       
from dateutil.parser import parse as date_parse
import uuid



class sourceCSV(TypedDict):
    name: str
    path: str
    schema_hint: None  #or dict[str, Any]
class MigrationState(TypedDict):
    run_id: str
    sources: list[sourceCSV]      
    target_schema: dict[str, Any]          
    status: Literal['pending', 'in_progress', 'completed', 'failed']
    profile_summaries: dict[str, Any]
    ontology: dict[str, Any]
    mapping_plan: dict[str, Any]
    validation_report: dict[str, Any]
    decision_log: dict[str, Any]
    checkpoint_id: str #path to checkpoint file





In [19]:
def save_checkpoint(state: MigrationState, checkpoint_dir: str = "checkpoints") -> str:
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint_id = f"cp_{uuid.uuid4().hex[:8]}"
    state["checkpoint_id"] = checkpoint_id

    checkpoint_path = os.path.join(checkpoint_dir, f"{checkpoint_id}.json")
    with open(checkpoint_path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

    return checkpoint_id

def load_checkpoint(checkpoint_id: str, checkpoint_dir: str = "checkpoints") -> MigrationState:
    checkpoint_path = os.path.join(checkpoint_dir, f"{checkpoint_id}.json")
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        return json.load(f)

#PROFILER

In [ ]:
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
PHONE_RE = re.compile(r"^\+?\d[\d\s\-\(\)]{7,}$")
INT_RE = re.compile(r"^[-+]?\d+$")
FLOAT_RE = re.compile(r"^[-+]?\d*\.\d+$|^[-+]?\d+\.\d*$")
DATE_HINT_RE = re.compile(
    r"(\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b)|"
    r"(\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b)|"
    r"(\b\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4}\b)|"
    r"(\b[A-Za-z]{3,9}\s+\d{1,2},\s+\d{2,4}\b)|"
    r"(\b[A-Za-z]{3,9}-\d{4}\b)"
)

#here u can see how strict or loose the regexes are. We can adjust them based on the data we see in the profiling step. For example, if we see a lot of phone numbers with extensions, we might want to allow for that in the regex. Or if we see a lot of dates in a specific format, we might want to add a regex for that format. The idea is to start with some basic regexes and then refine them as we learn more about the data.

In [ ]:



def normalize_value(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip() #this is for example to handle cases where there are leading or trailing spaces in the data, which can affect type inference. By stripping the whitespace, we can get a cleaner value to work with.

def safe_float(value: str) -> float | None:
    try:
        return float(value)
    except Exception:
        return None #this is to handle cases where we want to attempt to convert a string to a float, but it might not be a valid number. Instead of raising an exception, we return None to indicate that it couldn't be converted.


def looks_like_date_candidate(value: str) -> bool:
    """
    Only try date parsing when the string looks date-like.
    This reduces false positives like IDs, version numbers, or plain integers.
    """
    s = normalize_value(value)
    if not s:
        return False

    # Reject obvious pure numbers first
    if INT_RE.fullmatch(s) or FLOAT_RE.fullmatch(s):
        return False

    # Only attempt parse if it has date-ish shape
    if DATE_HINT_RE.search(s):
        return True

    return False


def parse_as_date(value: str) -> bool:
    """
    Uses dateutil as a flexible parser, but only after the value
    already looks date-like.
    """
    if not looks_like_date_candidate(value):
        return False

    try:
        date_parse(value, fuzzy=False)
        return True
    except Exception:
        return False


def infer_value_type(value: str) -> str:
    s = normalize_value(value)
    if s == "":
        return "null"

    if s.lower() in {"true", "false"}:
        return "boolean"

    if INT_RE.fullmatch(s):
        return "integer"

    if FLOAT_RE.fullmatch(s):
        return "float"

    if EMAIL_RE.fullmatch(s):
        return "email"

    if PHONE_RE.fullmatch(s):
        return "phone"

    if parse_as_date(s):
        return "date"

    return "string"


def profile_csv(csv_path: str, sample_size: int = 5) -> dict[str, Any]:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        headers = reader.fieldnames or []
        rows = list(reader)

    row_count = len(rows)
    profile: dict[str, Any] = {
        "file_path": csv_path,
        "row_count": row_count,
        "column_count": len(headers),
        "columns": {},
    }

    for col in headers:
        values = [row.get(col, "") for row in rows]
        non_null = [normalize_value(v) for v in values if normalize_value(v) != ""]
        null_count = len(values) - len(non_null)
        null_pct = round((null_count / row_count) * 100, 2) if row_count else 0.0

        type_counts: Counter[str] = Counter(infer_value_type(v) for v in non_null)
        sample_values = non_null[:sample_size]
        unique_count = len(set(non_null))

        # Evidence ratios
        total = len(non_null) if non_null else 1
        int_like = sum(1 for v in non_null if INT_RE.fullmatch(v))
        float_like = sum(1 for v in non_null if FLOAT_RE.fullmatch(v))
        email_like = sum(1 for v in non_null if EMAIL_RE.fullmatch(v))
        phone_like = sum(1 for v in non_null if PHONE_RE.fullmatch(v))
        date_like = sum(1 for v in non_null if parse_as_date(v))

        column_name_lower = col.lower()

        evidence = {
            "integer_like_ratio": round(int_like / total, 4),
            "float_like_ratio": round(float_like / total, 4),
            "email_like_ratio": round(email_like / total, 4),
            "phone_like_ratio": round(phone_like / total, 4),
            "date_like_ratio": round(date_like / total, 4),
        }

        # Infer a type only if evidence is strong
        inferred_type = "string"
        if evidence["date_like_ratio"] >= 0.9 or "date" in column_name_lower or "time" in column_name_lower:
            if evidence["date_like_ratio"] >= 0.6:
                inferred_type = "date"
        elif evidence["email_like_ratio"] >= 0.9 or "email" in column_name_lower:
            inferred_type = "email"
        elif evidence["phone_like_ratio"] >= 0.9 or "phone" in column_name_lower or "mobile" in column_name_lower:
            inferred_type = "phone"
        elif evidence["integer_like_ratio"] >= 0.9:
            inferred_type = "integer"
        elif evidence["float_like_ratio"] >= 0.9:
            inferred_type = "float"
        elif type_counts:
            inferred_type = type_counts.most_common(1)[0][0]

        column_profile: dict[str, Any] = {
            "null_count": null_count,
            "null_pct": null_pct,
            "unique_count": unique_count,
            "sample_values": sample_values,
            "type_distribution": dict(type_counts),
            "evidence": evidence,
            "inferred_type": inferred_type,
            "candidate_key": unique_count == row_count and null_count == 0 and row_count > 0,
        }

        # Numeric stats only when most values are numeric-like
        numeric_values = []
        for v in non_null:
            fv = safe_float(v)
            if fv is not None:
                numeric_values.append(fv)

        if numeric_values and len(numeric_values) / len(non_null) >= 0.9:
            column_profile["numeric_stats"] = {
                "min": min(numeric_values),
                "max": max(numeric_values),
                "mean": round(mean(numeric_values), 4),
            }

        if evidence["date_like_ratio"] >= 0.9:
            parsed_dates = []
            for v in non_null:
                try:
                    parsed_dates.append(date_parse(v, fuzzy=False))
                except Exception:
                    pass

            if parsed_dates:
                column_profile["date_stats"] = {
                    "min": min(parsed_dates).isoformat(),
                    "max": max(parsed_dates).isoformat(),
                }

        # String stats for plain text columns
        if inferred_type == "string" and non_null:
            lengths = [len(v) for v in non_null]
            column_profile["string_stats"] = {
                "min_length": min(lengths),
                "max_length": max(lengths),
                "avg_length": round(mean(lengths), 2),
            }

        profile["columns"][col] = column_profile

    return profile


def profiler_node(state: MigrationState) -> MigrationState:
    state["status"] = "in_progress"

    source = state["sources"][0]
    csv_path = source["path"]

    profile = profile_csv(csv_path)
    state["profile_summaries"][source["name"]] = profile

    state["decision_log"].append({
        "step": "profiler",
        "source": source["name"],
        "status": "completed"
    })

    return state